In [ ]:
# -*- coding: utf-8 -*-
"""TasNet + MR-STFT + Discriminator — On-the-fly loading — Only first 900 pairs per split"""

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import torchaudio
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────────────────────────────────────────
# Config
# ────────────────────────────────────────────────────────────────
class Config:
    SR = 52734
    DURATION = 3.0
    N_SAMPLES = int(SR * DURATION)

    ENC_CHANNELS = 384
    BOTTLE_CHANNELS = 192
    NUM_LAYERS = 4
    KERNEL_SIZE = 3
    DILATION_BASE = 2

    BATCH_SIZE = 6
    LEARNING_RATE_G = 3e-4
    LEARNING_RATE_D = 1e-4
    EPOCHS = 50              # ← as you want to finish in limited time
    PATIENCE = 20
    ADV_LAMBDA = 0.1
    MR_LAMBDA = 0.4

    DATASET_PATH = 'C:/Users/User/denoising_dataset/speedboat'  # ← change per category
    SAVE_DIR = 'C:/Users/User/trained_models/speedboat_tasnet_adv'
    LOG_DIR = 'C:/Users/User/logs/speedboat_tasnet_adv'

    MAX_FILES = 900          # ← NEW: limit to first 900 pairs

    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def __init__(self):
        Path(self.SAVE_DIR).mkdir(parents=True, exist_ok=True)
        Path(self.LOG_DIR).mkdir(parents=True, exist_ok=True)
        self.print_config()

    def print_config(self):
        print("="*70)
        print("TasNet + MR-STFT + Discriminator — Only first 900 pairs")
        print("="*70)
        print(f"Dataset: {self.DATASET_PATH}")
        print(f"Save dir: {self.SAVE_DIR}")
        print(f"Device: {self.DEVICE}")
        print(f"Batch size: {self.BATCH_SIZE}")
        print(f"Epochs: {self.EPOCHS}")
        print(f"Max files per split: {self.MAX_FILES}")
        print("="*70 + "\n")

config = Config()

# ────────────────────────────────────────────────────────────────
# Dataset — On-the-fly + limited to first 900 pairs
# ────────────────────────────────────────────────────────────────
class TasNetDatasetRAM(Dataset):
    def __init__(self, dataset_path, split='train'):
        self.root = Path(dataset_path) / split
        print(f"\nPre-loading {split.upper()} dataset into RAM (first {config.MAX_FILES} pairs)...")

        clean_files = sorted((self.root / 'clean').glob('*.wav'))
        mix_files   = sorted((self.root / 'mixture').glob('*.wav'))

        # Limit to first MAX_FILES pairs
        max_f = min(config.MAX_FILES, min(len(clean_files), len(mix_files)))
        clean_files = clean_files[:max_f]
        mix_files   = mix_files[:max_f]

        self.mixture_data = []
        self.clean_data   = []
        self.filenames    = []

        for c_path, m_path in tqdm(zip(clean_files, mix_files), total=max_f, desc=f"Loading {split}"):
            clean, _ = torchaudio.load(c_path)
            mix,   _ = torchaudio.load(m_path)

            # Normalize
            max_val = max(mix.abs().max(), clean.abs().max()) + 1e-8
            mix   = mix / max_val
            clean = clean / max_val

            # Fix length
            target = config.N_SAMPLES
            if clean.shape[1] > target:
                clean = clean[:, :target]
                mix   = mix[:, :target]
            else:
                clean = F.pad(clean, (0, target - clean.shape[1]))
                mix   = F.pad(mix, (0, target - mix.shape[1]))

            self.clean_data.append(clean.squeeze(0))
            self.mixture_data.append(mix.squeeze(0))
            self.filenames.append(m_path.stem)

        print(f"✓ Loaded {len(self.clean_data)} pairs into RAM for {split}")

    def __len__(self):
        return len(self.clean_data)

    def __getitem__(self, idx):
        return {
            'mixture': self.mixture_data[idx],
            'clean':   self.clean_data[idx],
            'filename': self.filenames[idx]
        }

# ────────────────────────────────────────────────────────────────
# Model, Discriminator, Losses, Training — unchanged
# ────────────────────────────────────────────────────────────────
class TasNetDenoiser(nn.Module):
    def __init__(self, enc_channels=384, bottle_channels=192, num_layers=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, enc_channels, kernel_size=16, stride=8, bias=False),
            nn.PReLU()
        )
        self.gln = nn.GroupNorm(1, enc_channels)
        self.separator = nn.ModuleList()
        for i in range(num_layers):
            dilation = 2 ** i
            self.separator.append(
                nn.Sequential(
                    nn.Conv1d(enc_channels, bottle_channels, 1),
                    nn.PReLU(),
                    nn.GroupNorm(1, bottle_channels),
                    nn.Conv1d(bottle_channels, bottle_channels,
                              kernel_size=3, padding=dilation, dilation=dilation,
                              groups=bottle_channels),
                    nn.PReLU(),
                    nn.GroupNorm(1, bottle_channels),
                    nn.Conv1d(bottle_channels, enc_channels, 1)
                )
            )
        self.lstm = nn.LSTM(enc_channels, enc_channels, bidirectional=True, batch_first=True)
        self.lstm_proj = nn.Linear(enc_channels*2, enc_channels)
        self.decoder = nn.ConvTranspose1d(enc_channels, 1, kernel_size=16, stride=8, bias=False)
        self.out_scale = nn.Parameter(torch.ones(1))

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        enc = self.encoder(x)
        latent = self.gln(enc)
        skip = 0
        for block in self.separator:
            res = block(latent)
            skip = skip + res
            latent = latent + res
        latent = latent.permute(0, 2, 1)
        lstm_out, _ = self.lstm(latent)
        latent = self.lstm_proj(lstm_out).permute(0, 2, 1)
        out = self.decoder(latent).squeeze(1)
        out = out * self.out_scale
        T = x.shape[-1]
        if out.shape[-1] != T:
            out = out[..., :T] if out.shape[-1] > T else F.pad(out, (0, T - out.shape[-1]))
        return out

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.utils.spectral_norm(nn.Conv1d(1, 64, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.utils.spectral_norm(nn.Conv1d(64, 128, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.utils.spectral_norm(nn.Conv1d(128, 256, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.utils.spectral_norm(nn.Conv1d(256, 1, 4, stride=1, padding=0)),
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.net(x)

def mr_stft_loss(est, target):
    loss = 0.0
    fft_sizes = [512, 1024, 2048]
    for n_fft in fft_sizes:
        window = torch.hann_window(n_fft).to(est.device)
        hop_length = n_fft // 4
        stft = lambda x: torch.stft(x, n_fft=n_fft, hop_length=hop_length,
                                    win_length=n_fft, window=window, return_complex=True)
        est_spec = stft(est)
        tgt_spec = stft(target)
        loss += F.l1_loss(est_spec.abs(), tgt_spec.abs())
        loss += F.l1_loss(est_spec.angle(), tgt_spec.angle()) * 0.05
    return loss / len(fft_sizes)

def si_snr_loss(est, target, eps=1e-8):
    est = est - est.mean(dim=1, keepdim=True)
    target = target - target.mean(dim=1, keepdim=True)
    dot = (est * target).sum(dim=1, keepdim=True)
    target_pow = torch.clamp((target**2).sum(dim=1, keepdim=True), min=eps)
    proj = dot * target / target_pow
    noise = est - proj
    ratio = 10 * torch.log10((proj.norm(dim=1)**2 + eps) / (noise.norm(dim=1)**2 + eps))
    return -ratio.mean()

def gan_loss_D(disc_real, disc_fake):
    real_loss = F.relu(1 - disc_real).mean()
    fake_loss = F.relu(1 + disc_fake).mean()
    return real_loss + fake_loss

def gan_loss_G(disc_fake):
    return -disc_fake.mean()

def train_step(G, D, real_clean, mix, optimizer_G, optimizer_D):
    optimizer_G.zero_grad()
    optimizer_D.zero_grad()
    fake_clean = G(mix)
    real_out = D(real_clean)
    fake_out = D(fake_clean.detach())
    d_loss = gan_loss_D(real_out, fake_out)
    d_loss.backward()
    optimizer_D.step()
    fake_out_G = D(fake_clean)
    g_gan_loss = gan_loss_G(fake_out_G)
    si_snr = si_snr_loss(fake_clean, real_clean)
    mr_loss = mr_stft_loss(fake_clean, real_clean)
    g_loss = si_snr + config.MR_LAMBDA * mr_loss + config.ADV_LAMBDA * g_gan_loss
    g_loss.backward()
    torch.nn.utils.clip_grad_norm_(G.parameters(), 5.0)
    optimizer_G.step()
    return g_loss.item(), d_loss.item(), si_snr.item()

def train_model():
    G = TasNetDenoiser().to(config.DEVICE)
    D = Discriminator().to(config.DEVICE)
    opt_G = optim.AdamW(G.parameters(), lr=config.LEARNING_RATE_G, weight_decay=1e-5)
    opt_D = optim.AdamW(D.parameters(), lr=config.LEARNING_RATE_D, weight_decay=1e-5)
    sch_G = optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=config.EPOCHS, eta_min=1e-6)
    sch_D = optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=config.EPOCHS, eta_min=1e-6)
    writer = SummaryWriter(config.LOG_DIR)
    best_val_sisnr = -float('inf')
    patience_counter = 0
    history = {'train_g_loss': [], 'val_sisnr': []}
    for epoch in range(1, config.EPOCHS + 1):
        G.train()
        D.train()
        train_g_loss = train_d_loss = train_sisnr = 0.0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch:03d}')
        for batch in pbar:
            mix = batch['mixture'].to(config.DEVICE)
            clean = batch['clean'].to(config.DEVICE)
            g_loss, d_loss, sisnr = train_step(G, D, clean, mix, opt_G, opt_D)
            train_g_loss += g_loss
            train_d_loss += d_loss
            train_sisnr += sisnr
            pbar.set_postfix({
                'G': f'{g_loss:.4f}',
                'D': f'{d_loss:.4f}',
                'SI-SNR': f'{sisnr:.2f}'
            })
        train_g_loss /= len(train_loader)
        train_sisnr /= len(train_loader)
        val_sisnr = validate(G, val_loader)
        sch_G.step()
        sch_D.step()
        history['train_g_loss'].append(train_g_loss)
        history['val_sisnr'].append(val_sisnr)
        writer.add_scalars('loss', {'G': train_g_loss, 'D': train_d_loss}, epoch)
        writer.add_scalar('SI-SNR/train', train_sisnr, epoch)
        writer.add_scalar('SI-SNR/val', val_sisnr, epoch)
        print(f"Epoch {epoch:03d} | G: {train_g_loss:.4f} | D: {train_d_loss:.4f} | "
              f"Train SI-SNR: {train_sisnr:.2f} | Val SI-SNR: {val_sisnr:.2f}")
        if val_sisnr > best_val_sisnr:
            best_val_sisnr = val_sisnr
            patience_counter = 0
            torch.save({
                'G_state_dict': G.state_dict(),
                'D_state_dict': D.state_dict(),
                'opt_G': opt_G.state_dict(),
                'val_sisnr': val_sisnr
            }, Path(config.SAVE_DIR) / 'best_model.pth')
            print("✓ Best model saved")
        else:
            patience_counter += 1
            if patience_counter >= config.PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break
    writer.close()
    return history

def validate(G, loader):
    G.eval()
    total_sisnr = 0.0
    with torch.no_grad():
        for batch in loader:
            mix = batch['mixture'].to(config.DEVICE)
            clean = batch['clean'].to(config.DEVICE)
            enhanced = G(mix)
            sisnr = -si_snr_loss(enhanced, clean).item()
            total_sisnr += sisnr
    return total_sisnr / len(loader)

# ────────────────────────────────────────────────────────────────
# Run
# ────────────────────────────────────────────────────────────────
train_set = TasNetDatasetRAM(config.DATASET_PATH, 'train')
val_set   = TasNetDatasetRAM(config.DATASET_PATH, 'val')
test_set  = TasNetDatasetRAM(config.DATASET_PATH, 'test')

train_loader = DataLoader(train_set, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=1, shuffle=False, num_workers=0)
history = train_model()

print("\n" + "═"*70)
print("TRAINING FINISHED")
print("═"*70)
print("Best model saved at:", Path(config.SAVE_DIR) / 'best_model.pth')
print("═"*70)

TasNet + MR-STFT + Discriminator — Only first 900 pairs
Dataset: C:/Users/User/denoising_dataset/speedboat
Save dir: C:/Users/User/trained_models/speedboat_tasnet_adv
Device: cuda
Batch size: 6
Epochs: 50
Max files per split: 900


Pre-loading TRAIN dataset into RAM (first 900 pairs)...


Loading train: 100%|████████████████████████████████████████████████████████████████| 900/900 [00:05<00:00, 173.96it/s]


✓ Loaded 900 pairs into RAM for train

Pre-loading VAL dataset into RAM (first 900 pairs)...


Loading val: 100%|██████████████████████████████████████████████████████████████████| 450/450 [00:02<00:00, 163.28it/s]


✓ Loaded 450 pairs into RAM for val

Pre-loading TEST dataset into RAM (first 900 pairs)...


Loading test: 100%|█████████████████████████████████████████████████████████████████| 452/452 [00:03<00:00, 138.45it/s]


✓ Loaded 452 pairs into RAM for test


Epoch 001:   0%|                                                                               | 0/150 [00:00<?, ?it/s]